<a href="https://colab.research.google.com/github/safakatakancelik/portfolio-public/blob/main/notebooks/building_attention_from_scratch/attention_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implementing Attention

1. Raw attention with word embeddings, no Q/K/V
2. Add Q, K, V matrices
3. Causal masking (set future positions to −∞ before softmax)
5. Multi-head attention (split dims across heads)
6. Training (parallel, masked) vs inference (sequential, KV cache)
7. Inspect attention matrices on ambiguous sentences
8. Rewrite as PyTorch nn.Module inside a transformer block (attention → residual → LayerNorm → FFN)

### Learning Goals
- understand raw attention vs learned Q/K/V projections
- see why √d_k scaling prevents softmax saturation
- understand causal masking for autoregressive generation
- use MHA as context for showing KV cache benefits
- compare training-time parallelism vs inference-time sequential decoding
- read attention maps to see what the model "looks at"
- bridge from numpy prototype to production-style PyTorch code


$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$


---

sources used:
https://machinelearningmastery.com/the-attention-mechanism-from-scratch/




on reasoning:

causal masking seems one of the limits of why these models are great at deductive reasoning and not abductive reasoning for example, but this is only looking at one perspective and one function. we need this functionality too.
this is based on temporal ordering

pattern matching, not reasoning: "doing sophisticated pattern matching and statistical prediction based on training data, not building an internal causal model

Passive Learning vs. Intervention

Hallucination and Spurious Correlations


---

Why Mask Instead of Truncating? Implementing not using future tokens
During training, entire sequence is processed at once in a single forward pass, not like inference, token per token.

We want to predict EVERY next token simultaneously:
this allows parallelism, and prevents immense sequential operations


In [1]:
!pip install torch
!pip install torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.5 MB/s eta 0:00:00


In [2]:

import numpy as np

In [23]:
np.random.seed(42)
d_model = 8
d_k = 8

### Step 1: Raw Attention

In [24]:


# building the vocabulary with random vectors
english_words = ["the", "cat", "sat", "on", "the", "mat"]

unique_words = dict.fromkeys(english_words)
vocab = {w: np.random.randn(d_model) for w in unique_words}


X = np.array([vocab[w] for w in english_words])  # (6, 8)


## Attention
attention_scores = X @ X.T / np.sqrt(d_k)  # (6, 8) @ (8, 6) -> (6, 6) attention score for each word pair

print("Tokens:", english_words)
print("\nScores shape:", attention_scores.shape)
print(np.round(attention_scores, 2))

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']

Scores shape: (6, 6)
[[ 2.19 -1.44 -1.61  0.08  2.19 -0.96]
 [-1.44  2.81  1.13  0.38 -1.44  1.98]
 [-1.61  1.13  2.89 -0.85 -1.61  0.37]
 [ 0.08  0.38 -0.85  2.13  0.08  0.03]
 [ 2.19 -1.44 -1.61  0.08  2.19 -0.96]
 [-0.96  1.98  0.37  0.03 -0.96  3.17]]


---


## Step 2: Add Q, K, V matrices

In [25]:
# initialize projection matrices to train
W_q = np.random.randn(d_model, d_k) * 0.01
W_k = np.random.randn(d_model, d_k) * 0.01
W_v = np.random.randn(d_model, d_k) * 0.01


# project embeddings into Q, K, V spaces
Q = X @ W_q
K = X @ W_k
V = X @ W_v

## Attention
attention_scores = Q @ K.T / np.sqrt(d_k)  # (6, 6)

print("Tokens:", english_words)
print("\nScores shape:", attention_scores.shape)
print(np.round(attention_scores, 5))

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']

Scores shape: (6, 6)
[[-5.80e-04  1.02e-03  2.60e-04  2.60e-04 -5.80e-04 -8.00e-05]
 [ 8.40e-04 -1.29e-03 -3.50e-04 -2.80e-04  8.40e-04 -6.10e-04]
 [-3.30e-04 -4.10e-04  5.90e-04 -3.60e-04 -3.30e-04 -2.70e-04]
 [ 2.10e-04 -2.60e-04 -3.60e-04  1.90e-04  2.10e-04 -1.30e-04]
 [-5.80e-04  1.02e-03  2.60e-04  2.60e-04 -5.80e-04 -8.00e-05]
 [ 3.40e-04 -6.80e-04  3.80e-04 -1.20e-03  3.40e-04 -1.40e-04]]
